# DataShield 360 · Minimarket Diva — Solución Big Data con Apache Spark
**Curso:** Base de Datos Avanzadas y Big Data (CIIN1487R) · UPN · 2026-2
**Evidencias:** E-02 (dataset procesado), E-06 (notebook), E-08 (resultados Big Data), E-09 (escenarios EXP-02 y EXP-03), E-10 (logs y figuras)

| Sección | Contenido |
|---|---|
| 0 | Entorno y reproducibilidad (versiones, semilla, hardware) |
| 1 | Ingesta del export crudo del POS (`ventas_pos_raw.csv`) |
| 2 | Perfilado de calidad del dataset **original** |
| 3 | Limpieza y transformación → dataset **procesado** (Parquet particionado) |
| 4 | Análisis con DataFrames, SparkSQL y RDD (resultados R1–R6) |
| 5 | **EXP-02** Escalabilidad: volumen (x1–x20), particiones (1–16) y formato (CSV vs Parquet) |
| 6 | **EXP-03** Comparativo pandas (un solo proceso) vs Spark |
| 7 | Exportación de resultados para Power BI y log de ejecución |

In [1]:
# 0. ENTORNO Y REPRODUCIBILIDAD ------------------------------------------------
import os, sys, time, json, platform, statistics, datetime as dt
import psutil, pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import *

SEMILLA   = 2026
RAW_CSV   = "../05_datos/bigdata/ventas_pos_raw.csv"     # E-01 dataset original
LAKE      = "lake"                                       # zona procesada (Parquet)
RES, FIG, LOG = "resultados", "figuras", "logs"
for d in (LAKE, RES, FIG, LOG): os.makedirs(d, exist_ok=True)

spark = (SparkSession.builder
         .appName("DataShield360-MinimarketDiva")
         .master("local[*]")                              # todos los núcleos del equipo
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

ENTORNO = {
    "fecha_ejecucion": dt.datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(), "spark": spark.version, "pandas": pd.__version__,
    "sistema": f"{platform.system()} {platform.release()}", "cpu": platform.processor() or platform.machine(),
    "nucleos_logicos": psutil.cpu_count(), "ram_gb": round(psutil.virtual_memory().total / 2**30, 1),
    "master": spark.sparkContext.master, "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
    "semilla": SEMILLA, "dataset": RAW_CSV, "tamano_mb": round(os.path.getsize(RAW_CSV) / 2**20, 2),
}
pd.Series(ENTORNO)

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=45813 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=


Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=45813 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/13 22:23:42 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/11/13 22:23:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/11/13 22:23:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


fecha_ejecucion                          2026-11-13T22:23:47
python                                               3.13.18
spark                                                  4.3.0
pandas                                                 3.0.5
sistema                                Linux 6.18.44-fc-v470
cpu                                                   x86_64
nucleos_logicos                                            2
ram_gb                                                   7.8
master                                              local[*]
shuffle_partitions                                         4
semilla                                                 2026
dataset               ../05_datos/bigdata/ventas_pos_raw.csv
tamano_mb                                               15.2
dtype: object

## 1. Ingesta (batch) del export del POS
Se define el **esquema explícito** (no se infiere) para que la lectura sea determinista y rápida.

In [2]:
ESQUEMA = StructType([
    StructField("id_detalle", IntegerType()), StructField("id_venta", IntegerType()),
    StructField("fecha_venta", StringType()), StructField("id_cliente", IntegerType()),
    StructField("metodo_pago", StringType()), StructField("id_empleado", IntegerType()),
    StructField("turno", StringType()), StructField("id_producto", IntegerType()),
    StructField("producto", StringType()), StructField("categoria", StringType()),
    StructField("cantidad", IntegerType()), StructField("precio_unitario", DoubleType()),
    StructField("costo_unitario", DoubleType()), StructField("estado_venta", StringType())])

t0 = time.perf_counter()
raw = spark.read.csv(RAW_CSV, header=True, schema=ESQUEMA, encoding="UTF-8")
n_raw = raw.count()
t_ingesta = time.perf_counter() - t0
print(f"Filas originales: {n_raw:,} | particiones: {raw.rdd.getNumPartitions()} | ingesta: {t_ingesta:.2f} s")
raw.printSchema()
raw.show(5, truncate=False)

Filas originales: 152,216 | particiones: 2 | ingesta: 3.13 s
root
 |-- id_detalle: integer (nullable = true)
 |-- id_venta: integer (nullable = true)
 |-- fecha_venta: string (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- metodo_pago: string (nullable = true)
 |-- id_empleado: integer (nullable = true)
 |-- turno: string (nullable = true)
 |-- id_producto: integer (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- costo_unitario: double (nullable = true)
 |-- estado_venta: string (nullable = true)



+----------+--------+-------------------+----------+-----------+-----------+------+-----------+----------------------------------+---------+--------+---------------+--------------+------------+
|id_detalle|id_venta|fecha_venta        |id_cliente|metodo_pago|id_empleado|turno |id_producto|producto                          |categoria|cantidad|precio_unitario|costo_unitario|estado_venta|
+----------+--------+-------------------+----------+-----------+-----------+------+-----------+----------------------------------+---------+--------+---------------+--------------+------------+
|93381     |46372   |2025-11-01 20:13:37|128       |Tarjeta    |6          |Noche |15         |Cusqueña Dorada 620 ml            |Cerveza  |6       |9.2            |7.0           |REGISTRADA  |
|86607     |43026   |2025-09-18 14:15:56|158       |Plin       |1          |Mañana|9          |Coca-Cola 3 L                     |Gaseosa  |1       |12.1           |9.8           |REGISTRADA  |
|128552    |63911   |2026-06-1

## 2. Perfilado de calidad del dataset original (tabla 6.1 del informe)

In [3]:
ts = F.to_timestamp("fecha_venta", "yyyy-MM-dd HH:mm:ss")
perfil = raw.agg(
    F.count("*").alias("registros"),
    (F.count("*") - F.countDistinct("id_detalle")).alias("duplicados_id_detalle"),
    F.sum(F.when(F.col("metodo_pago").isNull() | (F.trim("metodo_pago") == ""), 1).otherwise(0)).alias("metodo_pago_vacio"),
    F.sum(F.when(F.col("cantidad") <= 0, 1).otherwise(0)).alias("cantidad_no_valida"),
    F.sum(F.when(ts > F.lit("2026-12-31"), 1).otherwise(0)).alias("fecha_futura"),
    F.sum(F.when(F.col("categoria") != F.initcap(F.trim("categoria")), 1).otherwise(0)).alias("categoria_no_estandar"),
).toPandas().T.rename(columns={0: "original"})
celdas = n_raw * len(raw.columns)
vacias = raw.select([F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(c)
                     for c in raw.columns]).toPandas().sum(axis=1)[0]
perfil.loc["completitud_%"] = round(100 * (1 - vacias / celdas), 3)
perfil

/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,original
registros,152216.000
duplicados_id_detalle,1200.000
metodo_pago_vacio,150.000
cantidad_no_valida,120.000
fecha_futura,80.000
categoria_no_estandar,300.000
completitud_%,99.993


## 3. Limpieza y transformación → dataset procesado
Reglas (alineadas con el ETL de SQL Server):
- **Q1** estandarizar `categoria` (TRIM + Initcap) · **Q2** deduplicar por `id_detalle` (se conserva la fila con método de pago informado)
- **Q3** descartar cantidad ≤ 0 · **Q4** descartar fechas futuras · **Q5** imputar método de pago vacío → `NO REGISTRADO`
- **Q6** derivar `importe`, `costo`, `margen`, `anio`, `mes`, `dia_semana`, `hora`, `es_alcohol`
- Salida en **Parquet particionado por anio/mes** (lectura columnar y poda de particiones).

In [4]:
t0 = time.perf_counter()
w = Window.partitionBy("id_detalle").orderBy(F.when(F.trim(F.coalesce("metodo_pago", F.lit(""))) == "", 1).otherwise(0))
proc = (raw
    .withColumn("categoria", F.initcap(F.trim("categoria")))                                   # Q1
    .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")                        # Q2
    .filter(F.col("cantidad") > 0)                                                              # Q3
    .withColumn("ts", F.to_timestamp("fecha_venta", "yyyy-MM-dd HH:mm:ss"))
    .filter(F.col("ts") <= F.lit("2026-12-31"))                                                 # Q4
    .withColumn("metodo_pago", F.when(F.trim(F.coalesce("metodo_pago", F.lit(""))) == "", "NO REGISTRADO")
                               .otherwise(F.col("metodo_pago")))                                 # Q5
    .withColumn("importe", F.round(F.col("cantidad") * F.col("precio_unitario"), 2))            # Q6
    .withColumn("anio", F.year("ts")).withColumn("mes", F.month("ts"))
    .withColumn("dia_semana", F.date_format("ts", "E")).withColumn("hora", F.hour("ts"))
    .withColumn("es_alcohol", F.col("categoria").isin("Cerveza", "Ron", "Pisco", "Whiskey", "Vino", "Champán")))

> El costo se calcula con la proporción **pCompra / pVenta del catálogo** aplicada al precio histórico (misma regla R5 del ETL de SQL Server), mediante un *broadcast join* con el catálogo:

In [5]:
catalogo = (spark.read.csv("../05_datos/dw/DimProducto.csv", header=True, inferSchema=True)
            .select(F.col("IdProducto").alias("id_producto"), F.col("PrecioVenta").alias("p_venta"), F.col("PrecioCompra").alias("p_compra")))
proc = (proc.join(F.broadcast(catalogo), "id_producto")                              # broadcast join: tabla pequeña
            .withColumn("costo", F.round(F.col("importe") * F.col("p_compra") / F.col("p_venta"), 2))
            .withColumn("margen", F.round(F.col("importe") - F.col("costo"), 2))
            .drop("p_venta", "p_compra", "costo_unitario"))
(proc.repartition("anio").write.mode("overwrite").partitionBy("anio", "mes").parquet(f"{LAKE}/ventas_procesadas"))
t_limpieza = time.perf_counter() - t0

ventas = spark.read.parquet(f"{LAKE}/ventas_procesadas").cache()
n_proc = ventas.count()
validas = ventas.filter("estado_venta = 'REGISTRADA'")
lake_mb = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(f"{LAKE}/ventas_procesadas") for f in fs if f.endswith(".parquet")) / 2**20
print(f"Procesadas: {n_proc:,} | REGISTRADAS: {validas.count():,} | rechazadas: {n_raw - n_proc:,} | "
      f"tiempo limpieza+escritura: {t_limpieza:.2f} s | Parquet: {lake_mb:.2f} MB vs CSV {ENTORNO['tamano_mb']} MB")

Procesadas: 150,816 | REGISTRADAS: 148,959 | rechazadas: 1,400 | tiempo limpieza+escritura: 6.00 s | Parquet: 4.38 MB vs CSV 15.2 MB


In [6]:
# Tabla de validación original vs procesado (6.1) -----------------------------
perfil_p = ventas.agg(
    F.count("*").alias("registros"), (F.count("*") - F.countDistinct("id_detalle")).alias("duplicados_id_detalle"),
    F.sum(F.when(F.col("metodo_pago") == "", 1).otherwise(0)).alias("metodo_pago_vacio"),
    F.sum(F.when(F.col("cantidad") <= 0, 1).otherwise(0)).alias("cantidad_no_valida"),
    F.sum(F.when(F.col("ts") > F.lit("2026-12-31"), 1).otherwise(0)).alias("fecha_futura"),
    F.sum(F.when(F.col("categoria") != F.initcap(F.trim("categoria")), 1).otherwise(0)).alias("categoria_no_estandar"),
).toPandas().T.rename(columns={0: "procesado"})
calidad = perfil.join(perfil_p)
calidad.loc["completitud_%", "procesado"] = 100.0
calidad.loc["imputados_NO_REGISTRADO", ["original", "procesado"]] = [0, ventas.filter("metodo_pago = 'NO REGISTRADO'").count()]
calidad.to_csv(f"{RES}/E02_calidad_original_vs_procesado.csv")
# Reconciliación con el Data Warehouse (FactVentas generado por el ETL)
total_spark = validas.agg(F.round(F.sum("importe"), 2)).first()[0]
fact = pd.read_csv("../05_datos/dw/FactVentas.csv")
print(f"Importe REGISTRADO Spark: S/ {total_spark:,.2f} | DW FactVentas: S/ {fact.Importe.sum():,.2f} | "
      f"filas Spark {validas.count():,} vs DW {len(fact):,}")
calidad

/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Importe REGISTRADO Spark: S/ 4,880,319.40 | DW FactVentas: S/ 4,880,319.40 | filas Spark 148,959 vs DW 148,959


,original,procesado
registros,152216.000,150816.0
duplicados_id_detalle,1200.000,0.0
metodo_pago_vacio,150.000,0.0
cantidad_no_valida,120.000,0.0
fecha_futura,80.000,0.0
categoria_no_estandar,300.000,0.0
completitud_%,99.993,100.0
imputados_NO_REGISTRADO,0.000,149.0


## 4. Análisis (DataFrames + SparkSQL + RDD)

In [7]:
validas.createOrReplaceTempView("ventas")
# R1. Ranking de productos por importe (SparkSQL + función de ventana) ------------
r1 = spark.sql("""
    SELECT producto, categoria, SUM(cantidad) AS unidades, ROUND(SUM(importe),2) AS importe,
           ROUND(SUM(margen),2) AS margen, ROUND(100*SUM(margen)/SUM(importe),1) AS margen_pct,
           DENSE_RANK() OVER (ORDER BY SUM(importe) DESC) AS ranking
    FROM ventas GROUP BY producto, categoria ORDER BY ranking""").toPandas()
r1.to_csv(f"{RES}/R1_ranking_productos.csv", index=False)
r1.head(10)

/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,producto,categoria,unidades,importe,margen,margen_pct,ranking
0,Cristal 620 ml,Cerveza,81302,660120.9,178620.14,27.1,1
1,Pilsen Trujillo,Cerveza,45511,390934.7,108592.25,27.8,2
2,Cusqueña Dorada 620 ml,Cerveza,39459,358107.1,94237.28,26.3,3
3,Johnnie Walker Red,Whiskey,2236,203055.0,36336.12,17.9,4
4,Cerveza Pilsen 1 L retornable,Cerveza,18418,184810.1,47522.65,25.7,5
5,Corona 355 ml,Cerveza,25462,182499.4,51099.41,28.0,6
6,Pilsen Callao lata 355 ml x6,Cerveza,6979,173336.9,40001.33,23.1,7
7,Cristal lata 355 ml x6,Cerveza,7205,165086.7,37831.57,22.9,8
8,Jack Daniels,Whiskey,1150,159345.1,27473.30,17.2,9
9,Ron Cartavio Black,Ron,3656,157139.1,34920.19,22.2,10


In [8]:
# R1b. Top 3 por categoría (ventana particionada) ---------------------------------
r1b = spark.sql("""
    SELECT * FROM (SELECT categoria, producto, ROUND(SUM(importe),2) AS importe,
                   ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY SUM(importe) DESC) AS puesto
                   FROM ventas GROUP BY categoria, producto) WHERE puesto <= 3 ORDER BY categoria, puesto""").toPandas()
r1b.to_csv(f"{RES}/R1b_top3_por_categoria.csv", index=False); r1b.head(12)

/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,categoria,producto,importe,puesto
0,Cerveza,Cristal 620 ml,660120.9,1
1,Cerveza,Pilsen Trujillo,390934.7,2
2,Cerveza,Cusqueña Dorada 620 ml,358107.1,3
3,Champán,Champán Ricadonna,56550.6,1
4,Champán,Freixenet Carta Nevada,42157.5,2
5,Champán,Champán Riccadonna Ruby,41831.9,3
6,Cigarrillos,Lucky Strike,93679.5,1
7,Cigarrillos,Marlboro Rojo x20,54283.9,2
8,Cigarrillos,Hamilton Blue x20,38428.0,3
9,Gaseosa,Coca-Cola 3 L,104008.2,1


In [9]:
# R2. Estacionalidad: ventas mensuales, crecimiento interanual e índice estacional ---
r2 = (validas.groupBy("anio", "mes").agg(F.round(F.sum("importe"), 2).alias("importe"),
                                          F.countDistinct("id_venta").alias("tickets"))
      .orderBy("anio", "mes").toPandas())
r2["ticket_promedio"] = (r2.importe / r2.tickets).round(2)
r2["var_interanual_%"] = (100 * r2.groupby("mes").importe.pct_change()).round(1)
comp = r2[r2.anio.isin([2024, 2025])]
indice = (comp.groupby("mes").importe.mean() / comp.importe.mean()).round(3).rename("indice_estacional")
r2 = r2.merge(indice, on="mes"); r2.to_csv(f"{RES}/R2_estacionalidad_mensual.csv", index=False)
print("Índice estacional (1.00 = mes promedio):"); print(indice.to_string())

/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Índice estacional (1.00 = mes promedio):
mes
1     0.998
2     0.851
3     0.917
4     0.818
5     0.953
6     0.920
7     1.180
8     0.955
9     0.884
10    1.005
11    0.973
12    1.546


In [10]:
fig, ax = plt.subplots(figsize=(10, 4))
for a, g in r2.groupby("anio"):
    ax.plot(g.mes, g.importe / 1000, marker="o", label=str(a))
ax.set(title="R2. Ventas mensuales por año (miles de S/)", xlabel="Mes", ylabel="Miles de S/", xticks=range(1, 13))
ax.legend(); ax.grid(alpha=.3); fig.tight_layout(); fig.savefig(f"{FIG}/R2_tendencia_mensual.png", dpi=150)

In [11]:
# R3. Mapa de calor día x hora (decisión: turnos y personal) --------------------
orden = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]; etiquetas = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
r3 = validas.groupBy("dia_semana", "hora").agg(F.countDistinct("id_venta").alias("tickets")).toPandas()
dias = validas.select(F.to_date("ts").alias("d")).distinct().count() / 7
mat = r3.pivot(index="dia_semana", columns="hora", values="tickets").reindex(orden).fillna(0) / dias
mat.to_csv(f"{RES}/R3_tickets_dia_hora.csv")
fig, ax = plt.subplots(figsize=(10, 3.6)); im = ax.imshow(mat.values, aspect="auto", cmap="YlOrRd")
ax.set(xticks=range(len(mat.columns)), xticklabels=mat.columns, yticks=range(7), yticklabels=etiquetas,
       title="R3. Tickets promedio por día y hora", xlabel="Hora")
fig.colorbar(im, ax=ax, label="tickets/hora"); fig.tight_layout(); fig.savefig(f"{FIG}/R3_heatmap_dia_hora.png", dpi=150)
print("Hora pico:", mat.max().idxmax(), "h | día pico:", etiquetas[int(np.argmax(mat.sum(axis=1).values))])

/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Hora pico: 20 h | día pico: Sáb


In [12]:
# R4. Efecto de la ley seca 2026 (DataFrames): ventas del fin de semana electoral vs
#     el promedio de los 4 fines de semana anteriores -------------------------------
def finde(d0):
    return validas.filter((F.to_date("ts") >= F.lit(d0)) & (F.to_date("ts") <= F.date_add(F.lit(d0), 1)))
filas = []
for eleccion, sab in [("Elecciones 12-abr", "2026-04-11"), ("Segunda vuelta 07-jun", "2026-06-06")]:
    base = pd.Timestamp(sab)
    act = finde(sab).groupBy("es_alcohol").agg(F.sum("importe").alias("v")).toPandas().set_index("es_alcohol").v
    prev = []
    for k in range(1, 5):
        prev.append(finde(str((base - pd.Timedelta(weeks=k)).date())).groupBy("es_alcohol").agg(F.sum("importe").alias("v"))
                    .toPandas().set_index("es_alcohol").v)
    prev = pd.concat(prev, axis=1).mean(axis=1)
    for alc in (True, False):
        filas.append({"evento": eleccion, "tipo": "Alcohol" if alc else "No alcohol", "ventas_finde": round(act.get(alc, 0), 2),
                      "promedio_4_findes_previos": round(prev.get(alc, 0), 2),
                      "variacion_%": round(100 * (act.get(alc, 0) / prev.get(alc, 1) - 1), 1)})
r4 = pd.DataFrame(filas); r4.to_csv(f"{RES}/R4_impacto_ley_seca.csv", index=False); r4

/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,evento,tipo,ventas_finde,promedio_4_findes_previos,variacion_%
0,Elecciones 12-abr,Alcohol,0.0,11048.37,-100.0
1,Elecciones 12-abr,No alcohol,4400.2,1672.88,163.0
2,Segunda vuelta 07-jun,Alcohol,155.0,11685.37,-98.7
3,Segunda vuelta 07-jun,No alcohol,4238.5,1734.40,144.4


In [13]:
# R5. Canasta: ¿quién compra cerveza también compra hielo? (soporte, confianza, lift)
tickets = validas.groupBy("id_venta").agg(F.collect_set("categoria").alias("cats")).cache()
N = tickets.count()
def sop(*cs): return tickets.filter(" AND ".join([f"array_contains(cats, '{c}')" for c in cs])).count() / N
reglas = []
for a, b in [("Cerveza", "Hielo"), ("Ron", "Gaseosa"), ("Whiskey", "Hielo"), ("Pisco", "Gaseosa"), ("Cerveza", "Cigarrillos")]:
    sa, sb, sab = sop(a), sop(b), sop(a, b)
    reglas.append({"regla": f"{a} -> {b}", "soporte_%": round(100 * sab, 2), "confianza_%": round(100 * sab / sa, 1), "lift": round(sab / (sa * sb), 2)})
r5 = pd.DataFrame(reglas).sort_values("lift", ascending=False); r5.to_csv(f"{RES}/R5_canasta.csv", index=False); r5

,regla,soporte_%,confianza_%,lift
0,Cerveza -> Hielo,18.79,30.1,1.09
4,Cerveza -> Cigarrillos,4.78,7.7,0.78
2,Whiskey -> Hielo,1.17,21.3,0.77
1,Ron -> Gaseosa,3.33,25.5,0.76
3,Pisco -> Gaseosa,2.78,25.6,0.76


In [14]:
# R6. Pareto de clientes frecuentes y adopción de pagos digitales -----------------
cli = (validas.filter("id_cliente <> 13").groupBy("id_cliente").agg(F.sum("importe").alias("importe"))
       .orderBy(F.desc("importe")).toPandas())
cli["acum_%"] = 100 * cli.importe.cumsum() / cli.importe.sum()
top20 = cli.head(int(len(cli) * .2)).importe.sum() / cli.importe.sum() * 100
pagos = (validas.groupBy("anio", "metodo_pago").agg(F.countDistinct("id_venta").alias("tickets")).toPandas()
         .pivot(index="anio", columns="metodo_pago", values="tickets").fillna(0))
pagos_pct = (100 * pagos.div(pagos.sum(axis=1), axis=0)).round(1); pagos_pct.to_csv(f"{RES}/R6_metodos_pago_pct.csv")
print(f"El 20 % de clientes frecuentes genera el {top20:.1f} % de su importe.")
pagos_pct

/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


El 20 % de clientes frecuentes genera el 61.9 % de su importe.


metodo_pago,Efectivo,NO REGISTRADO,Plin,Tarjeta,Transferencia,Yape
anio,,,,,,
2024,37.4,0.2,10.3,18.1,4.1,29.9
2025,32.8,0.2,11.1,17.6,3.9,34.4
2026,28.0,0.2,11.5,18.5,3.9,37.9


In [15]:
# Operaciones sobre RDD (Tema 7, semana 7): unidades por categoría con map/reduceByKey
rdd = validas.select("categoria", "cantidad").rdd
unidades = (rdd.map(lambda r: (r.categoria, r.cantidad)).reduceByKey(lambda a, b: a + b)
               .sortBy(lambda kv: -kv[1]).collect())
pd.DataFrame(unidades, columns=["categoria", "unidades"])

,categoria,unidades
0,Cerveza,233771
1,Gaseosa,33960
2,Hielo,25756
3,Ron,12274
4,Pisco,10051
5,Cigarrillos,9051
6,Vino,8486
7,Whiskey,5042
8,Champán,2651


In [16]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
t10 = r1.head(10).iloc[::-1]
axs[0].barh(t10.producto, t10.importe / 1000, color="#2a6f97"); axs[0].set(title="R1. Top 10 productos (miles de S/)", xlabel="Miles de S/")
pagos_pct.plot(kind="bar", stacked=True, ax=axs[1], colormap="tab20c"); axs[1].set(title="R6. Mezcla de métodos de pago (% tickets)", ylabel="%", xlabel="Año")
axs[1].legend(fontsize=7, loc="lower right"); fig.tight_layout(); fig.savefig(f"{FIG}/R1_R6_ranking_pagos.png", dpi=150)

## 5. EXP-02 · Escalabilidad de la solución Big Data
**Carga de trabajo W** (idéntica en todos los escenarios): leer → filtrar REGISTRADA → agregar importe y tickets por año-mes-categoría → ranking de productos con ventana → recoger resultado.
**Variables independientes:** (a) volumen x1, x5, x10, x20 · (b) particiones 1, 2, 4, 8, 16 con x10 · (c) formato CSV vs Parquet con x10.
**Variable dependiente:** tiempo total (s) y throughput (filas/s). **Controles:** mismo equipo, `local[*]`, caché limpia antes de cada corrida, 1 calentamiento descartado, **3 repeticiones**.

In [17]:
base = spark.read.parquet(f"{LAKE}/ventas_procesadas").drop("ts")
ESCALAS = [1, 5, 10, 20]
for k in ESCALAS:                                       # preparación (no se mide): datasets sintéticos escalados
    ruta = f"{LAKE}/escala_x{k}"
    if not os.path.exists(ruta):
        rep = spark.range(k).withColumnRenamed("id", "copia")
        (base.crossJoin(rep).withColumn("id_detalle", F.col("id_detalle") + F.col("copia") * 10_000_000)
             .withColumn("id_venta", F.col("id_venta") + F.col("copia") * 10_000_000).drop("copia")
             .write.mode("overwrite").parquet(ruta))
if not os.path.exists(f"{LAKE}/escala_x10_csv"):
    spark.read.parquet(f"{LAKE}/escala_x10").write.mode("overwrite").option("header", True).csv(f"{LAKE}/escala_x10_csv")
filas_escala = {k: spark.read.parquet(f"{LAKE}/escala_x{k}").count() for k in ESCALAS}
filas_escala

{1: 150816, 5: 754080, 10: 1508160, 20: 3016320}

In [18]:
def carga_spark(df):
    v = df.filter("estado_venta = 'REGISTRADA'").withColumn("importe", F.col("cantidad") * F.col("precio_unitario"))
    agg = v.groupBy("anio", "mes", "categoria").agg(F.sum("importe").alias("importe"), F.countDistinct("id_venta").alias("tickets"))
    rank = (v.groupBy("producto").agg(F.sum("importe").alias("importe"))
             .withColumn("r", F.dense_rank().over(Window.orderBy(F.desc("importe")))).filter("r <= 10"))
    return len(agg.collect()) + len(rank.collect())

def medir(nombre, leer, particiones=None, reps=3):
    spark.catalog.clearCache(); carga_spark(leer())                       # calentamiento
    tiempos = []
    for _ in range(reps):
        spark.catalog.clearCache()
        t0 = time.perf_counter()
        df = leer()
        if particiones: df = df.repartition(particiones)
        carga_spark(df); tiempos.append(time.perf_counter() - t0)
    return tiempos

registros = []
def registrar(exp, escenario, filas, tiempos, **extra):
    m, s = statistics.mean(tiempos), statistics.stdev(tiempos)
    registros.append({"experimento": exp, "escenario": escenario, "filas": filas, "reps": len(tiempos),
                      "media_s": round(m, 3), "desv_s": round(s, 3), "cv_%": round(100 * s / m, 1),
                      "min_s": round(min(tiempos), 3), "max_s": round(max(tiempos), 3),
                      "throughput_filas_s": int(filas / m), "tiempos": [round(t, 3) for t in tiempos], **extra})
    print(f"{exp} | {escenario:<28} | {filas:>10,} filas | {m:6.2f} ± {s:.2f} s | {int(filas/m):>10,} filas/s")

spark.conf.set("spark.sql.shuffle.partitions", "4")
for k in ESCALAS:                                                          # (a) volumen
    registrar("EXP-02a", f"Spark Parquet x{k}", filas_escala[k], medir(f"x{k}", lambda k=k: spark.read.parquet(f"{LAKE}/escala_x{k}")))

EXP-02a | Spark Parquet x1             |    150,816 filas |   0.78 ± 0.07 s |    193,315 filas/s


EXP-02a | Spark Parquet x5             |    754,080 filas |   1.23 ± 0.14 s |    611,364 filas/s


EXP-02a | Spark Parquet x10            |  1,508,160 filas |   1.80 ± 0.11 s |    840,080 filas/s


EXP-02a | Spark Parquet x20            |  3,016,320 filas |   2.80 ± 0.10 s |  1,076,885 filas/s


In [19]:
for p in [1, 2, 4, 8, 16]:                                                 # (b) particiones con x10
    spark.conf.set("spark.sql.shuffle.partitions", str(p))
    registrar("EXP-02b", f"x10 con {p} particiones", filas_escala[10],
              medir(f"p{p}", lambda: spark.read.parquet(f"{LAKE}/escala_x10"), particiones=p), particiones=p)
spark.conf.set("spark.sql.shuffle.partitions", "4")
esquema_csv = spark.read.parquet(f"{LAKE}/escala_x10").schema                # (c) formato con x10
registrar("EXP-02c", "x10 formato CSV", filas_escala[10],
          medir("csv", lambda: spark.read.csv(f"{LAKE}/escala_x10_csv", header=True, schema=esquema_csv)))
registrar("EXP-02c", "x10 formato Parquet", filas_escala[10], medir("pq", lambda: spark.read.parquet(f"{LAKE}/escala_x10")))

EXP-02b | x10 con 1 particiones        |  1,508,160 filas |   2.85 ± 0.05 s |    529,222 filas/s


EXP-02b | x10 con 2 particiones        |  1,508,160 filas |   3.98 ± 0.46 s |    379,055 filas/s


EXP-02b | x10 con 4 particiones        |  1,508,160 filas |   3.26 ± 0.04 s |    462,705 filas/s


EXP-02b | x10 con 8 particiones        |  1,508,160 filas |   3.30 ± 0.24 s |    457,055 filas/s


EXP-02b | x10 con 16 particiones       |  1,508,160 filas |   3.47 ± 0.19 s |    434,687 filas/s


EXP-02c | x10 formato CSV              |  1,508,160 filas |   3.93 ± 0.04 s |    383,847 filas/s


EXP-02c | x10 formato Parquet          |  1,508,160 filas |   1.53 ± 0.08 s |    984,975 filas/s


## 6. EXP-03 · pandas (un proceso, en memoria) vs Spark (distribuido)
Hipótesis H2: con volúmenes pequeños pandas es más rápido (Spark paga el costo de planificación y *shuffle*); al crecer el volumen la diferencia se reduce y Spark mantiene un tiempo casi lineal y no queda limitado a la RAM de un proceso.

In [20]:
def carga_pandas(ruta):
    df = pd.read_parquet(ruta, columns=["estado_venta", "cantidad", "precio_unitario", "anio", "mes", "categoria", "id_venta", "producto"])
    v = df[df.estado_venta == "REGISTRADA"].assign(importe=lambda d: d.cantidad * d.precio_unitario)
    agg = v.groupby(["anio", "mes", "categoria"]).agg(importe=("importe", "sum"), tickets=("id_venta", "nunique"))
    rank = v.groupby("producto").importe.sum().rank(method="dense", ascending=False)
    return len(agg) + int((rank <= 10).sum())

for k in ESCALAS:
    ruta = f"{LAKE}/escala_x{k}"
    carga_pandas(ruta); tiempos = []                                       # calentamiento
    for _ in range(3):
        t0 = time.perf_counter(); carga_pandas(ruta); tiempos.append(time.perf_counter() - t0)
    rss = psutil.Process().memory_info().rss / 2**20
    registrar("EXP-03", f"pandas Parquet x{k}", filas_escala[k], tiempos, rss_mb=round(rss))
res = pd.DataFrame(registros); res.to_csv(f"{RES}/EXP02_EXP03_resultados.csv", index=False)
res.drop(columns="tiempos")

EXP-03 | pandas Parquet x1            |    150,816 filas |   0.06 ± 0.01 s |  2,676,527 filas/s


EXP-03 | pandas Parquet x5            |    754,080 filas |   0.23 ± 0.02 s |  3,270,343 filas/s


EXP-03 | pandas Parquet x10           |  1,508,160 filas |   0.43 ± 0.04 s |  3,520,069 filas/s


EXP-03 | pandas Parquet x20           |  3,016,320 filas |   0.96 ± 0.01 s |  3,151,901 filas/s


,experimento,escenario,filas,reps,media_s,desv_s,cv_%,min_s,max_s,throughput_filas_s,particiones,rss_mb
0,EXP-02a,Spark Parquet x1,150816,3,0.780,0.069,8.9,0.700,0.826,193315,NaN,NaN
1,EXP-02a,Spark Parquet x5,754080,3,1.233,0.140,11.4,1.128,1.393,611364,NaN,NaN
2,EXP-02a,Spark Parquet x10,1508160,3,1.795,0.112,6.2,1.703,1.920,840080,NaN,NaN
3,EXP-02a,Spark Parquet x20,3016320,3,2.801,0.098,3.5,2.727,2.912,1076885,NaN,NaN
4,EXP-02b,x10 con 1 particiones,1508160,3,2.850,0.051,1.8,2.816,2.909,529222,1.0,NaN
5,EXP-02b,x10 con 2 particiones,1508160,3,3.979,0.464,11.7,3.535,4.461,379055,2.0,NaN
6,EXP-02b,x10 con 4 particiones,1508160,3,3.259,0.037,1.1,3.218,3.289,462705,4.0,NaN
7,EXP-02b,x10 con 8 particiones,1508160,3,3.300,0.238,7.2,3.061,3.536,457055,8.0,NaN
8,EXP-02b,x10 con 16 particiones,1508160,3,3.470,0.190,5.5,3.322,3.685,434687,16.0,NaN
9,EXP-02c,x10 formato CSV,1508160,3,3.929,0.039,1.0,3.891,3.969,383847,NaN,NaN


In [21]:
# Validación: ambos motores devuelven el MISMO total (x1)
v1 = spark.read.parquet(f"{LAKE}/escala_x1").filter("estado_venta='REGISTRADA'").agg(F.sum(F.col("cantidad")*F.col("precio_unitario"))).first()[0]
p1 = pd.read_parquet(f"{LAKE}/escala_x1").query("estado_venta=='REGISTRADA'").eval("cantidad*precio_unitario").sum()
print(f"Total Spark {v1:,.2f} | total pandas {p1:,.2f} | diferencia {abs(v1-p1):.6f}")

Total Spark 4,880,319.40 | total pandas 4,880,319.40 | diferencia 0.000000


In [22]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
a = res[res.experimento == "EXP-02a"]; c = res[res.experimento == "EXP-03"]
axs[0].errorbar(a.filas / 1e6, a.media_s, yerr=a.desv_s, marker="o", capsize=4, label="Spark")
axs[0].errorbar(c.filas / 1e6, c.media_s, yerr=c.desv_s, marker="s", capsize=4, label="pandas")
axs[0].set(title="EXP-02a / EXP-03: tiempo vs volumen", xlabel="Millones de filas", ylabel="segundos"); axs[0].legend(); axs[0].grid(alpha=.3)
b = res[res.experimento == "EXP-02b"]
axs[1].errorbar(b.particiones, b.media_s, yerr=b.desv_s, marker="o", capsize=4, color="#c1121f")
axs[1].set(title=f"EXP-02b: particiones (x10, {ENTORNO['nucleos_logicos']} núcleos)", xlabel="particiones", ylabel="segundos", xscale="log", xticks=[1, 2, 4, 8, 16], xticklabels=[1, 2, 4, 8, 16]); axs[1].grid(alpha=.3)
f = res[res.experimento == "EXP-02c"]
axs[2].bar(["CSV", "Parquet"], f.media_s, yerr=f.desv_s, capsize=6, color=["#adb5bd", "#2a9d8f"])
axs[2].set(title="EXP-02c: formato de almacenamiento (x10)", ylabel="segundos")
fig.tight_layout(); fig.savefig(f"{FIG}/EXP02_EXP03_escalabilidad.png", dpi=150)

## 7. Exportación para Power BI (punto de integración BI ↔ Big Data) y log de ejecución

In [23]:
# Resultados agregados que Power BI importa como tablas adicionales
(validas.groupBy("anio", "mes", "categoria", "metodo_pago", "turno")
        .agg(F.round(F.sum("importe"), 2).alias("importe"), F.round(F.sum("margen"), 2).alias("margen"),
             F.countDistinct("id_venta").alias("tickets"))
        .toPandas().to_csv(f"{RES}/powerbi_resumen_spark.csv", index=False))
LOG_RUN = {**ENTORNO, "filas_originales": n_raw, "filas_procesadas": n_proc, "filas_rechazadas": n_raw - n_proc,
           "t_ingesta_s": round(t_ingesta, 3), "t_limpieza_escritura_s": round(t_limpieza, 3),
           "parquet_mb": round(lake_mb, 2), "importe_registrado": total_spark,
           "experimentos": registros, "fin": dt.datetime.now().isoformat(timespec="seconds")}
with open(f"{LOG}/run_log_{dt.datetime.now():%Y%m%d_%H%M}.json", "w", encoding="utf-8") as fh:
    json.dump(LOG_RUN, fh, ensure_ascii=False, indent=2, default=str)
print("Log guardado. Archivos de resultados:", sorted(os.listdir(RES)))
spark.stop()

/usr/local/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Log guardado. Archivos de resultados: ['E02_calidad_original_vs_procesado.csv', 'EXP02_EXP03_resultados.csv', 'R1_ranking_productos.csv', 'R1b_top3_por_categoria.csv', 'R2_estacionalidad_mensual.csv', 'R3_tickets_dia_hora.csv', 'R4_impacto_ley_seca.csv', 'R5_canasta.csv', 'R6_metodos_pago_pct.csv', 'powerbi_resumen_spark.csv']
